# Lesson 27 — Machine Learning for Optimization

## Learning objectives

1. Distinguish *learning to branch*, *learning to cut*, *learning primal heuristics*.
2. Recognize the role of GNNs for instance representation {cite:p}`Gasse2019`.
3. Understand decision-focused vs predict-then-optimize learning {cite:p}`Elmachtoub2022,Wilder2019`.
4. Critically read a "ML beats Gurobi" paper.

## 1. The taxonomy

{cite:p}`Bengio2021` organize ML-for-optimization into:

- **End-to-end learning**: predict the solution from instance features. Limited to easy problems.
- **Configuring** combinatorial algorithms: choose hyperparameters per instance.
- **Interleaved with B&B**: learn branching, cuts, heuristics inside the tree.

## 2. Learning to branch

Replace strong branching (Lesson 14) with a fast classifier trained to imitate it. {cite:p}`Gasse2019` use a GNN over the variable-constraint bipartite graph.

**Training:** supervised — at each B&B node, log strong-branching scores, train classifier to predict argmax. **Inference:** use the classifier instead of strong branching, getting strong-branching quality at fraction of the cost.

Reported speedups: 2–5x on benchmark MILPs.

## 3. Learning primal heuristics

Train a model to *propose* good integer-feasible incumbents from the LP relaxation. Often used early in B&B. {cite:p}`Bengio2021` survey several approaches.

## 4. Decision-focused learning

When the *output* of a predictive model is fed into an optimizer, training the predictor to minimize prediction error is suboptimal — what matters is the *decision quality*. {cite:p}`Wilder2019,Mandi2020,Elmachtoub2022` develop *decision-focused* training: differentiate the optimizer (Lesson 26) and propagate gradients.

## 5. Reading the literature critically

ML-for-optimization papers vary widely in claim quality. Checklist:

- **Compared to what?** "Beats Gurobi default" without solver tuning is weak.
- **Generalization?** Trained and tested on the same instance distribution? Or transfers?
- **Compute cost?** Often the ML method's training cost is omitted.
- **Statistical reporting?** Confidence intervals on benchmarks are rare in ML papers.

For a sober view, see {cite:p}`Lu2024,Sandrin2025`. The field is rapidly evolving; today's "breakthrough" is often tomorrow's "moderate improvement".

## References
{cite:p}`Bengio2021,Gasse2019,Elmachtoub2022,Wilder2019,Mandi2020`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
# A real knapsack solve to anchor the discussion. discopt's stable
# branching_policy currently exposes "fractional" only; "gnn" is a future
# hook intended for ML-driven rules. Custom branching callbacks are not
# yet a stable public API — the exercises below treat them as a research
# extension.
import numpy as np, discopt as do

m = do.Model("knapsack20")
n = 20
rng = np.random.default_rng(0)
v = rng.integers(1, 30, n); w = rng.integers(1, 20, n); W = int(0.4 * w.sum())
x = m.binary("x", shape=(n,))
m.subject_to(w @ x <= W); m.maximize(v @ x)
r = m.solve(branching_policy="fractional")
print("nodes:", r.node_count, "  z*:", r.objective)
